In [ ]:
!pip install git+https://github.com/lightshing/wildlife-datasets@develop
!pip install git+https://github.com/lightshing/wildlife-tools

In [2]:
import os
import pandas as pd
import timm
import torchvision.transforms as T
import imgaug as ia
import imgaug.augmenters as iaa
import torch
from tqdm import tqdm

from PIL import Image
from torchvision.transforms import ToPILImage
from wildlife_datasets.datasets import AnimalCLEF2025
from wildlife_datasets import datasets, splits
from torch.utils.data import DataLoader
from sklearn.preprocessing import LabelEncoder

In [3]:
# name = 'hf-hub:BVRA/MegaDescriptor-L-384'
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# mega = timm.create_model(name, num_classes=0, pretrained=True)
# print(device)
# mega = mega.to(device)

In [4]:
root = '/kaggle/input/animal-clef-2025'
# root = '/Users/tingshuoliang/Downloads/CNN_test/animal-clef-2025'
new_root = '/kaggle/working/aug'
# new_root = '/Users/tingshuoliang/Downloads/CNN_test/testt'

In [5]:
# transform_mega_ori = timm.data.create_transform(
#     **timm.data.resolve_data_config(mega.pretrained_cfg)
# )
# transform_mega_ori

In [6]:
ia.seed(333)
transform_base_aug = T.Compose([
    iaa.Sequential([
        iaa.Sometimes(0.7, iaa.MultiplyBrightness((0.8, 1.3))),
        iaa.Sometimes(0.3, iaa.GammaContrast((0.7, 1.8))),
        iaa.Sometimes(0.8, iaa.CLAHE(clip_limit=(1, 6), tile_grid_size_px=(8, 8))),
        iaa.Sometimes(0.2, iaa.AllChannelsCLAHE(clip_limit=(1, 6), tile_grid_size_px=(4, 8))),
        
        iaa.Sometimes(0.3, iaa.AddToHueAndSaturation((-13, 13))),
        iaa.Sometimes(0.6,
            iaa.WithColorspace(
                to_colorspace="HSV",
                from_colorspace="RGB",
                children=iaa.WithChannels(
                    0,
                    iaa.Add((-10, 10))
                )
            )
        ),
        
        iaa.Sometimes(0.6,iaa.AdditivePoissonNoise(lam=(0, 5))),
        
        iaa.Sometimes(0.3,
            iaa.Affine(
                translate_percent={"x": (-0.1, 0.1), "y": (-0.1, 0.1)},
                mode='constant',
                cval=0
            )
        ),
        iaa.Sometimes(0.4,
            iaa.Affine(
                rotate=(-30, 30),
                mode='constant',
                cval=0
            )
        ),
        iaa.Sometimes(0.6,
            iaa.Affine(
                scale={"x": (0.8, 1.2), "y": (0.8, 1.2)},
                mode='constant',
                cval=0
            )
        ),
        iaa.Sometimes(0.8,iaa.Fliplr(0.5)),
        
        iaa.Sometimes(0.1, iaa.GaussianBlur(sigma=(0.0, 1.0))),
        iaa.Sometimes(0.3, iaa.Sharpen(alpha=(0.0, 0.5), lightness=(0.6, 1.4)))
    ], random_order=True).augment_image,
    ])

In [7]:
# transform_display_aug = T.Compose([
#     *transform_base_aug.transforms,
#     lambda img_np: Image.fromarray(img_np)
#     ])

In [8]:
transform_aug = T.Compose([
    *transform_base_aug.transforms,
    T.ToTensor(),
    # *transform_mega_ori.transforms
    ])

In [9]:
dataset = AnimalCLEF2025(root, transform=transform_aug, load_label=True)
dataset.metadata[['dataset', 'split']].value_counts()

dataset           split   
SeaTurtleID2022   database    8729
LynxID2025        database    2957
SalamanderID2025  database    1388
LynxID2025        query        946
SalamanderID2025  query        689
SeaTurtleID2022   query        500
Name: count, dtype: int64

### For individuals with images less than 5, replicate existing

In [10]:
df = dataset.df

identity_counts = df['identity'].value_counts()

rows_to_append = []
for identity, count in identity_counts.items():
    if count < 5:
        rows = df[(df['identity'] == identity) & (df['split'] == 'database')]
        num_to_add = 5 - count
        repeated_rows = pd.concat([rows]*num_to_add, ignore_index=True).iloc[:num_to_add]
        rows_to_append.append(repeated_rows)

if rows_to_append:
    df = pd.concat([df] + rows_to_append, ignore_index=True)

df['image_id'] = range(len(df))

dataset.df = df
dataset.metadata = df

验证

In [11]:
dataset.metadata[['dataset', 'split']].value_counts()

dataset           split   
SeaTurtleID2022   database    8821
SalamanderID2025  database    3064
LynxID2025        database    3006
                  query        946
SalamanderID2025  query        689
SeaTurtleID2022   query        500
Name: count, dtype: int64

### aug & save

In [15]:
csv_data = dataset.df

In [16]:
if not os.path.exists(new_root):
    os.makedirs(new_root)


to_pil = ToPILImage()

for i in tqdm(range(len(dataset)), desc="Processing images", unit="imgs"):
    tensor = dataset[i][0]
    row = csv_data.iloc[i]
    
    original_path = row["path"]
    
    dir_path = os.path.dirname(original_path)
    
    output_dir = os.path.join(new_root, dir_path)
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    new_filename = f"{i}.jpg"
    
    output_path = os.path.join(output_dir, new_filename)
    
    img = to_pil(tensor)
    img.save(output_path)
    
    new_relative_path = os.path.join(dir_path, new_filename)
    csv_data.iloc[i, csv_data.columns.get_loc("path")] = new_relative_path

Processing images: 100%|██████████| 17026/17026 [44:51<00:00,  6.33imgs/s]


In [17]:
csv_out_path = os.path.join(new_root, "metadata.csv")
csv_data.to_csv(csv_out_path, index=False, encoding='utf-8-sig')